[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/extensor_representations/extensor_representations.ipynb)

# Extensor representations

A multivector acts on a state by multiplication. The states need not fill the whole algebra: a smaller space can carry the action while still distinguishing every multivector and preserving every product. A projector selects such states, and leaving their independent coefficients open gives the representation as an extensor. This notebook builds the action, prints what each blade does, recovers multivectors from their actions, and constructs the smallest faithful real representations for every nondegenerate signature through dimension six.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
from numga import Algebra
from examples.math.extensor_representations import core

In [ ]:
# Two directions squaring to plus one. Every coefficient and operation is exact.
ga = Algebra("x+y+")
mv = ga.exact.multivector
Full = ga.gatype.full()
State = ga.gatype(ga.subspace("1 y"))
state_dimension = len(State.output_subspace)

## 1. Keep a smaller space of states

Multiplying on the right by \((1+x)/2\) selects states that are unchanged by another right multiplication by \(x\). Left multiplication by any multivector preserves that property. These states form a left ideal: they are closed under every left action. The coefficients of \(1\) and \(y\) suffice to describe them, since multiplication by the projector supplies the linked \(x\) and \(xy\) coefficients. Reading a state back out keeps those two coefficients and doubles them, since the projector halved them. Reading out after embedding returns every state, and embedding after reading out returns every element of the ideal. A multivector's action on the states embeds a state, multiplies it on the left and reads it back out.

In [ ]:
multivector = mv.x + 2 * mv.y + mv.xy

projector = (1 + mv.x) / 2
# Into the ideal: the projector fills in the coefficients of x and xy.
state_embedding = State * projector                                      # [] Full <- State
# Out of the ideal: the coefficients of 1 and y, doubled since the projector halved them.
state_readout = (2 * Full).cast(State)                                   # [] State <- Full
# A multivector's action: embed the state, multiply on the left, read the state back out.
embedder = state_readout(Full * state_embedding)                         # [] State <- (Full, State)

# The readout undoes the embedding, and the embedding undoes the readout on the ideal.
state_round_trip_residual = state_readout(state_embedding) - State       # [] State <- State
ideal = Full * projector                                                 # [] Full <- Full
ideal_round_trip_residual = state_embedding(state_readout(ideal)) - ideal   # [] Full <- Full

# The embedder takes a multivector; its State slot stays open in the representation.
representation = embedder(multivector)                                   # [] State <- State

# Read the multivector back out of its action.
recovered = core.readout(embedder, representation, Full)                 # [] Full

print("multivector: Full")
print(multivector.coefficient_table())
print("extensor representation: State <- State")
print(representation.coefficient_table())
print("recovered multivector: Full")
print(recovered.coefficient_table())

## 2. Every blade has an action

The embedder leaves the multiplying multivector open as well as the state. It has two slots: one for an element of the whole algebra and one for the state it acts on. Filling the first slot gives that multivector's unary representation; its state slot remains open. Filling both slots applies the representation to a state. The four basis blades give four independent actions, and every multivector gives their weighted sum.

In matrix notation these actions are real \(2\times2\) matrices, and the representation satisfies \(\rho(ab)=\rho(a)\rho(b)\).

In [ ]:
# Each blade acts by multiplication inside the ideal, followed by the same readout.
# The unary maps have output blades on rows and input blades on columns.
for blade in Full.output_subspace.masks:
    print(f"{ga.blade_name(blade)}:\n{embedder(mv.blade(blade)).coefficient_table()}\n")

In [ ]:
state = 1 + 2 * mv.y
other_multivector = 3 + mv.x - mv.y

representation = embedder(multivector)                                   # [] State <- State
other_representation = embedder(other_multivector)                       # [] State <- State
product_representation = embedder(multivector * other_multivector)       # [] State <- State

# Composition of the two actions is multiplication in the algebra.
composition_residual = representation(other_representation) - product_representation

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("action on the state:")
print(representation(state).coefficient_table())
print("product:")
print((multivector * other_multivector).coefficient_table())
print("successive actions:")
print(representation(other_representation(state)).coefficient_table())
print("product acting once:")
print(product_representation(state).coefficient_table())

## 3. Recover the multivector

An action retains all the multivector's coefficients, and one contraction reads them out. The coefficient of a blade is how much of the action is that blade's action: compose the action with the blade's inverse action and trace over the state, divided by the number of state components. The code composes the action with the action of an open multivector and traces; solving against the scalar product then replaces each blade by its inverse, which is the blade up to sign. Leaving both algebra inputs open gives the round trip: the readout after the representation is the identity on the algebra.

This readout is the adjoint of the representation, with actions paired by their normalized trace and multivectors by their scalar product. The representation preserves those pairings, so its adjoint is also its left inverse.

In matrix notation, with $\gamma_A$ the matrix of blade $A$ on $d$ state components, the readout of a matrix $M$ is the expansion $a_A = \operatorname{tr}(\gamma_A^{-1} M)/d$, and the round trip reads $\operatorname{tr}(\gamma_A^{-1}\gamma_B)/d = \delta_{AB}$.

In [ ]:
# The action composed with each blade's inverse action, traced over the state.
recovered = core.readout(embedder, representation, Full)                 # [] Full
coefficient_residual = recovered - multivector

pairing = (1 * Full).scalar_product(Full)                                # [] Scalar <- (Full, Full)

# Compose two open actions and trace their State slot.
# The normalized result is exactly the scalar product of their algebra inputs.
trace_pairing = embedder(Full, embedder).trace(0, 3)
round_trip = pairing.solve(trace_pairing / state_dimension)              # [] Full <- Full
round_trip_residual = round_trip - Full

print("multivector:")
print(multivector.coefficient_table())
print("extensor representation:")
print(representation.coefficient_table())
print("recovered multivector:")
print(recovered.coefficient_table())

## 4. Project an arbitrary action

A faithful representation need not fill the space of all actions on its states. In three Euclidean dimensions, eight multivector coefficients act on four real state components; a general linear action there has sixteen coefficients. The same trace readout projects any such action onto the represented algebra: the representation after its adjoint is the orthogonal projection onto represented actions under the trace pairing. An action already in the representation returns unchanged. For an action outside it, the remainder pairs to zero with every represented action.

In [ ]:
ga = Algebra("x+y+z+")
mv = ga.exact.multivector
Full = ga.gatype.full()
State = ga.gatype.even()

projector = (1 + mv.z) / 2
state_embedding = State * projector                                      # [] Full <- State
state_readout = (2 * Full).cast(State)                                   # [] State <- Full
embedder = state_readout(Full * state_embedding)                         # [] State <- (Full, State)

# Keep only the scalar component of a state. This is a valid state map, but no
# multivector produces that action: every nonzero represented action has even rank.
arbitrary_action = (1 * State).select[0].cast(State)
projected = core.readout(embedder, arbitrary_action, Full)               # [] Full
projected_representation = embedder(projected)                           # [] State <- State
remainder = arbitrary_action - projected_representation
projection_residual = embedder(Full, remainder).trace(0, 2)

print("State:", State)
print("supplied action:", arbitrary_action.gatype)
print(arbitrary_action.coefficient_table())
print("projected multivector:", projected.gatype)
print(projected.coefficient_table())
print("represented action:", projected_representation.gatype)
print(projected_representation.coefficient_table())
print("remainder:", remainder.gatype)
print(remainder.coefficient_table())

## 5. Construct the state space for each signature

Several commuting blades that square to plus one give several commuting projectors; each halves the state space. Products of those blades divide the full basis into disjoint groups. Keeping one representative per group gives independent state coordinates, and casting back multiplies by the number of projectors' combined copies. The layout helper selects these blades and coordinates; the extensor construction below is the same for every signature. Where the algebra has two independent blocks, the selection retains both, so neither can disappear from the action.

In matrix notation these are the smallest faithful real matrix representations. Complex and quaternionic entries are counted by their real state components. The [Clifford Algebra Explorer](https://coreality.cc/clifford/) lists the corresponding algebra classifications and projector constructions.

In [ ]:
# The upper dimension of the signature table.
dimensions = 6
signatures = tuple((positive, dimension - positive)
                   for dimension in range(dimensions + 1)
                   for positive in range(dimension, -1, -1))

print(f"{'signature':12s} {'algebra components':>18s} {'real state components':>23s}")
for positive, negative in signatures:
    ga = Algebra((positive, negative, 0))
    mv = ga.exact.multivector
    Full = ga.gatype.full()
    State, involutions = core.spinor_layout(ga)
    multiplicity = 2 ** len(involutions)

    # The choices are discrete blade data. The projector and all subsequent
    # operations are exact extensors, with no coefficient conversion.
    projector = mv.scalar()
    for blade in involutions:
        projector = projector * (1 + mv.blade(blade)) / 2

    state_embedding = State * projector
    state_readout = (multiplicity * Full).cast(State)
    embedder = state_readout(Full * state_embedding)

    print(f"{str((positive, negative)):12s} {len(Full.output_subspace):18d}"
          f" {len(State.output_subspace):23d}")

Both the three-dimensional Euclidean algebra and the spacetime algebra of signature $(3,1)$ act faithfully on four real state components. The blade labels of their generator actions below name the chosen state coordinates: rows are output components and columns are input components.

In [ ]:
representative_signatures = ("x+y+z+", "x+y+z+t-")

for signature in representative_signatures:
    ga = Algebra(signature)
    mv = ga.exact.multivector
    Full = ga.gatype.full()
    State, involutions = core.spinor_layout(ga)
    multiplicity = 2 ** len(involutions)

    projector = mv.scalar()
    for blade in involutions:
        projector = projector * (1 + mv.blade(blade)) / 2
    state_readout = (multiplicity * Full).cast(State)
    embedder = state_readout(Full * (State * projector))

    print(f"Cl{ga.pqr[:2]}")
    for blade in ga.basis_vector_masks:
        print(f"\n{ga.blade_name(blade)}:\n{embedder(mv.blade(blade)).coefficient_table()}")
    print()

In [ ]:
# checks
print("State readout after embedding less the identity:\n" + state_round_trip_residual.coefficient_table())
print("Embedding after state readout, on the ideal, less the identity:\n" + ideal_round_trip_residual.coefficient_table())
print("Composition less multiplication:\n" + composition_residual.coefficient_table())
print("Recovered multivector less its input:\n" + coefficient_residual.coefficient_table())
print("Recovery less the identity:\n" + round_trip_residual.coefficient_table())
print("Trace pairing of the projection remainder:\n" + projection_residual.coefficient_table())